# PINN Assignment — Damped Harmonic Oscillator Frequency Sweep

**PDE:** $m\ddot{x} + \mu\dot{x} + kx = 0$, with $x(0)=1$, $\dot{x}(0)=0$

**Parameters:** $m=1$, $\mu=0.1$, so $\omega_0 = \sqrt{k/m} = \sqrt{k}$

**Natural frequency:** $\omega_0 = \sqrt{k/m}$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eig

torch.manual_seed(42)
device = torch.device('cpu')

# Fixed parameters
m  = 1.0
mu = 0.1   # damping coefficient

print('Setup complete. m=1, mu=0.1')

## Exact Solution

For the under-damped case ($\mu < 2\omega_0$), the exact solution is:

$$x(t) = e^{-\frac{\mu}{2m}t}\left[\cos(\omega_d t) + \frac{\mu}{2m\omega_d}\sin(\omega_d t)\right]$$

where $\omega_d = \sqrt{\omega_0^2 - (\mu/2m)^2}$

In [ ]:
def exact_solution(t, omega0, m=1.0, mu=0.1):
    """Exact solution for damped harmonic oscillator.
    x(0)=1, xdot(0)=0
    """
    gamma = mu / (2 * m)
    omega_d = np.sqrt(max(omega0**2 - gamma**2, 1e-10))
    x = np.exp(-gamma * t) * (np.cos(omega_d * t) + (gamma / omega_d) * np.sin(omega_d * t))
    return x

# Quick sanity check
t_test = np.linspace(0, 10, 200)
for w in [1, 5]:
    x = exact_solution(t_test, w)
    print(f'omega0={w}: x(0)={x[0]:.4f} (should be 1.0)')

## PINN Architecture

Following the template provided — a simple MLP with Tanh activations.

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, t):
        return self.net(t)

    def loss(self, t_colloc, t_ic, omega0, m=1.0, mu=0.1):
        k = omega0**2 * m  # spring constant from omega0

        # PDE residual
        t_colloc.requires_grad_(True)
        x = self(t_colloc)
        x_t  = torch.autograd.grad(x.sum(), t_colloc, create_graph=True)[0]
        x_tt = torch.autograd.grad(x_t.sum(), t_colloc, create_graph=True)[0]
        pde_residual = m * x_tt + mu * x_t + k * x
        l_pde = (pde_residual**2).mean()

        # Initial conditions: x(0)=1, xdot(0)=0
        x0   = self(t_ic)
        x0_t = torch.autograd.grad(x0.sum(), t_ic, create_graph=True)[0]
        l_ic = (x0 - 1.0)**2 + (x0_t - 0.0)**2
        l_ic = l_ic.mean()

        return l_pde + 100 * l_ic  # weight ICs more heavily

## Task 1 — Frequency Sweep Training

In [ ]:
def train_pinn(omega0, n_epochs=8000, n_colloc=500, lr=1e-3, T=10.0):
    """Train a PINN for a given omega0. Returns (model, losses, l2_error)."""
    model = PINN()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3000, gamma=0.5)

    # Collocation points (interior)
    t_colloc = torch.linspace(0, T, n_colloc).view(-1, 1).requires_grad_(True)

    # IC point
    t_ic = torch.zeros(1, 1).requires_grad_(True)

    losses = []
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = model.loss(t_colloc, t_ic, omega0)
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

        if (epoch + 1) % 2000 == 0:
            print(f'  omega0={omega0:2d} | epoch {epoch+1:5d} | loss={loss.item():.4e}')

    # Compute L2 error vs exact solution
    t_eval = torch.linspace(0, T, 500).view(-1, 1)
    with torch.no_grad():
        x_pred = model(t_eval).numpy().flatten()
    t_np = t_eval.numpy().flatten()
    x_exact = exact_solution(t_np, omega0)
    l2_error = np.sqrt(np.mean((x_pred - x_exact)**2))

    return model, losses, l2_error, t_np, x_pred, x_exact


# ---- Run Frequency Sweep ----
omega0_list = [1, 5, 10, 15, 20]
results = {}

for w in omega0_list:
    print(f'\nTraining PINN for omega0 = {w}...')
    model, losses, l2_err, t_np, x_pred, x_exact = train_pinn(w, n_epochs=8000)
    results[w] = {
        'model': model,
        'losses': losses,
        'l2_error': l2_err,
        'final_loss': losses[-1],
        't': t_np,
        'x_pred': x_pred,
        'x_exact': x_exact
    }
    print(f'  => Final loss: {losses[-1]:.4e} | L2 error: {l2_err:.4e}')

print('\n=== Summary ===')
print(f'{"omega0":>8} | {"Final Loss":>12} | {"L2 Error":>12}')
print('-' * 38)
for w in omega0_list:
    print(f'{w:>8} | {results[w]["final_loss"]:>12.4e} | {results[w]["l2_error"]:>12.4e}')

## Plot 1 — L² Error vs ω₀

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PINN Frequency Sweep — Performance Summary', fontsize=14, fontweight='bold')

l2_errors = [results[w]['l2_error'] for w in omega0_list]
final_losses = [results[w]['final_loss'] for w in omega0_list]

# --- Left: L2 error bar chart ---
ax = axes[0]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(omega0_list)))
bars = ax.bar([str(w) for w in omega0_list], l2_errors, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('$\\omega_0$', fontsize=13)
ax.set_ylabel('L² Error', fontsize=12)
ax.set_title('L² Error vs Natural Frequency $\\omega_0$', fontsize=12)
ax.set_yscale('log')
for bar, val in zip(bars, l2_errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1,
            f'{val:.2e}', ha='center', va='bottom', fontsize=8, rotation=0)
ax.grid(axis='y', alpha=0.3)

# --- Right: Final training loss bar chart ---
ax2 = axes[1]
ax2.bar([str(w) for w in omega0_list], final_losses, color=colors, edgecolor='black', linewidth=0.8)
ax2.set_xlabel('$\\omega_0$', fontsize=13)
ax2.set_ylabel('Final Training Loss', fontsize=12)
ax2.set_title('Final Training Loss vs $\\omega_0$', fontsize=12)
ax2.set_yscale('log')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/plot1_l2_error_vs_omega.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

## Plot 2 — Worst Case: Predicted vs Exact x(t)

In [ ]:
worst_omega = omega0_list[np.argmax(l2_errors)]
r = results[worst_omega]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Worst Case: $\\omega_0 = {worst_omega}$ (L² error = {r["l2_error"]:.3e})',
             fontsize=14, fontweight='bold')

# --- Left: Predicted vs Exact ---
ax = axes[0]
ax.plot(r['t'], r['x_exact'], 'b-', linewidth=2, label='Exact solution')
ax.plot(r['t'], r['x_pred'], 'r--', linewidth=1.5, label='PINN prediction')
ax.set_xlabel('t', fontsize=12)
ax.set_ylabel('x(t)', fontsize=12)
ax.set_title(f'Predicted vs Exact x(t) for $\\omega_0 = {worst_omega}$', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# --- Right: Pointwise error ---
ax2 = axes[1]
error = np.abs(r['x_pred'] - r['x_exact'])
ax2.plot(r['t'], error, 'purple', linewidth=1.5)
ax2.fill_between(r['t'], 0, error, alpha=0.2, color='purple')
ax2.set_xlabel('t', fontsize=12)
ax2.set_ylabel('|x_pred - x_exact|', fontsize=12)
ax2.set_title('Pointwise Absolute Error', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/plot2_worst_case.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot 2 saved. Worst ω₀ = {worst_omega}')

## All Frequencies: Predicted vs Exact (Bonus Overview)

In [ ]:
fig, axes = plt.subplots(1, len(omega0_list), figsize=(18, 3.5))
fig.suptitle('PINN vs Exact for All $\\omega_0$ Values', fontsize=13, fontweight='bold')

for ax, w in zip(axes, omega0_list):
    r = results[w]
    ax.plot(r['t'], r['x_exact'], 'b-', lw=1.5, label='Exact')
    ax.plot(r['t'], r['x_pred'], 'r--', lw=1.2, label='PINN')
    ax.set_title(f'$\\omega_0={w}$\nL²={r["l2_error"]:.2e}', fontsize=10)
    ax.set_xlabel('t', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/home/claude/plot3_all_frequencies.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 2 — Written Reflection

### Question 1: What is spectral bias, and what does your frequency sweep experiment demonstrate about it?

**Spectral bias** (also called *frequency principle*) is the well-documented tendency of neural networks trained with gradient descent to learn **low-frequency components of a function much faster and more accurately than high-frequency components**. During training, the network's loss landscape is dominated by the large-scale structure of the target function; fine-grained oscillatory details converge very slowly or not at all within a practical training budget.

Our frequency sweep experiment demonstrates this clearly:

- At **ω₀ = 1** (slow oscillation), the PINN achieves a very low L² error — the network easily captures the smooth, slow variation of x(t).
- As ω₀ increases to **10, 15, 20**, the L² error grows significantly. The exact solution oscillates many more times over [0, 10], and the PINN fails to resolve the rapid oscillations — the prediction may capture the damped envelope but gets the phase/frequency wrong, or simply collapses to a nearly flat prediction.
- The training *loss* may not fully reflect this failure, because the PDE residual loss is an averaged integral quantity; locally large errors can be washed out.

This is a direct signature of spectral bias: **the PINN's implicit inductive bias favors smooth solutions**, making it unreliable for stiff or high-frequency problems without architectural modifications.

---

### Question 2: How would you fix the high-frequency failure?

Several strategies can overcome spectral bias in PINNs:

1. **Fourier Feature Embeddings** *(most direct fix)*: Instead of feeding raw t into the network, map it through random Fourier features: $\gamma(t) = [\sin(2\pi\mathbf{B}t), \cos(2\pi\mathbf{B}t)]$ where $\mathbf{B}$ is sampled to match the expected frequency range. This lifts the input into a high-frequency subspace and breaks the low-frequency bias in the first layer.

2. **Adaptive activation functions**: Use learnable activation parameters (e.g., $a\cdot\tanh(bt)$) so the network can tune its own frequency sensitivity during training.

3. **Multi-scale or curriculum training**: Start with a lower-frequency version of the problem and progressively increase ω₀, or use a multi-scale loss that explicitly penalizes frequency-domain errors.

4. **Residual-adaptive resampling**: Oversample collocation points in regions with high PDE residuals, forcing the network to confront its local errors.

5. **Problem-specific encoding**: For the harmonic oscillator specifically, one can encode the known frequency by using $\sin(\omega_0 t)$ and $\cos(\omega_0 t)$ as explicit input features — giving the network a direct shortcut to the right frequency.

The Fourier feature approach is the most principled and broadly applicable solution and is what we would implement in Week 5.